# SDT Model Compilation and Testing Workflow

This notebook provides step-by-step instructions for compiling and testing the SDT (Signal Detection Theory) JAGS module.


## Prerequisites

Ensure you have the required dependencies installed:
- JAGS 4.3.0+
- ONNX Runtime 1.23.2+ (Linux x64)
- py2jags 0.1.0+
- Python 3.11+ with scikit-learn


## Step 1: Generate Module Code


In [12]:
!cd /home/jovyan/project && ./scripts/generate-module models/sdt.jnnx


Found files:
  Metadata: models/sdt.jnnx/metadata.json
  ONNX: models/sdt.jnnx/model.onnx
  Scalers: models/sdt.jnnx/scalers.pkl

Metadata loaded: sdt_emulator
Scalers loaded: 4 parameters

✓ ONNX Runtime found in tmp/onnxruntime-linux-x64-1.23.2

Output directory: tmp/sdt.jnnx_build

Generating module code...
Copied ONNX model to: tmp/sdt.jnnx_build/model.onnx
Generated: tmp/sdt.jnnx_build/sdt_emulator.cc

Generating Makefile...
Generated: tmp/sdt.jnnx_build/Makefile

Module generation complete!

To compile and install:
  cd tmp/sdt.jnnx_build
  make
  sudo make install


## Step 2: Compile the Module


In [13]:
!cd tmp/sdt.jnnx_build && make


/bin/bash: line 1: cd: tmp/sdt.jnnx_build: No such file or directory


In [14]:
!cd tmp/sdt.jnnx_build && sudo make install


/bin/bash: line 1: cd: tmp/sdt.jnnx_build: No such file or directory


## Step 3: Test the Module


In [15]:
import py2jags
import onnxruntime as ort
import numpy as np

# Test SDT function
model_string = '''
model {
    result <- sdt(0.0, 0.0)
    dummy ~ dnorm(0, 1)
}
'''

result = py2jags.run_jags(
    model_string=model_string,
    data_dict={'n': 1},
    nchains=1, nsamples=1, nadapt=0, nburnin=0,
    monitorparams=['result'],
    modules=['sdt_emulator'],
    verbosity=0
)

print('SDT output:', [result.get_samples(f'result_{i+1}')[0] for i in range(2)])


SDT output: [0.481909, 1.0]


## Step 4: Validate Against Python ONNX


In [16]:
import onnxruntime as ort
import numpy as np

# Load ONNX model with absolute path
session = ort.InferenceSession('/home/jovyan/project/models/sdt.jnnx/model.onnx')
test_input = np.array([[0.0, 0.0]], dtype=np.float32)
result = session.run(['output'], {'input': test_input})
print('Python ONNX output:', result[0][0].tolist())


Python ONNX output: [0.48190850019454956, 0.9999998807907104]


## Step 5: Compare Results


In [17]:
# Run both tests and compare
import py2jags
import onnxruntime as ort
import numpy as np

# JAGS evaluation
model_string = '''
model {
    result <- sdt(0.0, 0.0)
    dummy ~ dnorm(0, 1)
}
'''

jags_result = py2jags.run_jags(
    model_string=model_string,
    data_dict={'n': 1},
    nchains=1, nsamples=1, nadapt=0, nburnin=0,
    monitorparams=['result'],
    modules=['sdt_emulator'],
    verbosity=0
)

jags_output = [jags_result.get_samples(f'result_{i+1}')[0] for i in range(2)]

# Python ONNX evaluation with absolute path
session = ort.InferenceSession('/home/jovyan/project/models/sdt.jnnx/model.onnx')
test_input = np.array([[0.0, 0.0]], dtype=np.float32)
onnx_result = session.run(['output'], {'input': test_input})
onnx_output = onnx_result[0][0].tolist()

# Compare results
print('JAGS output:  ', jags_output)
print('Python output:', onnx_output)
print('Differences:  ', [abs(jags_output[i] - onnx_output[i]) for i in range(2)])

# Check if results match within machine precision
max_diff = max([abs(jags_output[i] - onnx_output[i]) for i in range(2)])
if max_diff < 1e-6:
    print('SUCCESS: Results match within machine precision')
else:
    print('ERROR: Results differ significantly')


JAGS output:   [0.481909, 1.0]
Python output: [0.48190850019454956, 0.9999998807907104]
Differences:   [4.998054504157246e-07, 1.1920928955078125e-07]
SUCCESS: Results match within machine precision


## Expected Results

Both outputs should match within machine precision (differences < 1e-6). The SDT model outputs 2 values representing:

1. Hit rate
2. False alarm rate
